# 02 - Preprocesamiento de Datos

Pipeline de transformacion para modelado

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import os
import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv('../data/raw/telco_customer_churn.csv')
print(f'Shape original: {df.shape}')
df.head()

Shape original: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,CUST-00000,Male,0,No,No,66,Yes,No,No,Yes,...,Yes,No internet service,No,No,Two year,No,Mailed check,89.06,4247.82,Yes
1,CUST-00001,Female,1,No,No,24,Yes,No,Fiber optic,No internet service,...,No,No,No,No,Two year,Yes,Electronic check,85.55,5849.52,No
2,CUST-00002,Male,0,No,No,60,Yes,No,DSL,Yes,...,Yes,No,No internet service,No,Month-to-month,Yes,Bank transfer (automatic),22.61,8270.10,No
3,CUST-00003,Male,0,Yes,No,25,Yes,No phone service,No,No internet service,...,No internet service,Yes,No,No,Month-to-month,No,Electronic check,55.51,3589.81,No
4,CUST-00004,Male,0,Yes,No,33,Yes,No,Fiber optic,No,...,No internet service,No internet service,No internet service,No,Month-to-month,Yes,Mailed check,71.07,4124.59,Yes


In [3]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

In [4]:
df['avg_monthly_per_tenure'] = df['TotalCharges'] / (df['tenure'] + 1)

service_cols = ['PhoneService', 'InternetService', 'OnlineSecurity',
                'OnlineBackup', 'DeviceProtection', 'TechSupport',
                'StreamingTV', 'StreamingMovies']
df['num_services'] = df[service_cols].apply(
    lambda x: sum(1 for v in x if v in ['Yes', 'DSL', 'Fiber optic']), axis=1
)

In [5]:
df = df.drop('customerID', axis=1)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

In [6]:
X = df.drop('Churn', axis=1)
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {X_train.shape[0]}, Test: {X_test.shape[0]}')

Train: 5634, Test: 1409


In [7]:
X_train_enc = pd.get_dummies(X_train, drop_first=True)
X_test_enc = pd.get_dummies(X_test, drop_first=True)

X_train_enc, X_test_enc = X_train_enc.align(X_test_enc, join='left', axis=1, fill_value=0)

# Reset index to avoid alignment issues
X_train_enc = X_train_enc.reset_index(drop=True)
X_test_enc = X_test_enc.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

print(f'Features: {X_train_enc.shape[1]}')

Features: 32


In [8]:
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges',
            'avg_monthly_per_tenure', 'num_services']
num_cols = [c for c in num_cols if c in X_train_enc.columns]

scaler = StandardScaler()
X_train_enc[num_cols] = scaler.fit_transform(X_train_enc[num_cols])
X_test_enc[num_cols] = scaler.transform(X_test_enc[num_cols])

In [9]:
os.makedirs('../data/processed', exist_ok=True)

train_data = pd.concat([X_train_enc, y_train], axis=1)
test_data = pd.concat([X_test_enc, y_test], axis=1)

train_data.to_csv('../data/processed/train.csv', index=False)
test_data.to_csv('../data/processed/test.csv', index=False)

print(f'Train: {train_data.shape}, NaN: {train_data.isna().sum().sum()}')
print(f'Test: {test_data.shape}, NaN: {test_data.isna().sum().sum()}')

Train: (5634, 33), NaN: 0
Test: (1409, 33), NaN: 0
